## Prerequisites
1. An Azure AI Foundry project with a deployed model (e.g. `gpt-4o`).
2. Fill in `.env` in this folder:
   - `PROJECT_ENDPOINT` — your Foundry project endpoint
   - `MODEL_DEPLOYMENT_NAME` — the deployed model to use when creating a new agent
   - `AGENT_NAME` — optional, the **name** of an existing agent to reuse (not a GUID). Leave blank to create a new agent each run.
   - `FOUNDRY_API_KEY` - If you have a foundry key
3. Authenticate with Azure (any one of these works with `DefaultAzureCredential`):
   - Run `az login` in a terminal, or
   - Sign in via VS Code's Azure extension, or
   - Set `AZURE_CLIENT_ID` / `AZURE_CLIENT_SECRET` / `AZURE_TENANT_ID` env vars for a service principal.


In [3]:
from dotenv import load_dotenv
load_dotenv()

True

# Calling the model directly (no agent)

Here, `get_openai_client()` is called with **no `agent_name`**, so it points at the project's general model endpoint instead of any specific agent. That means:
- You must pass `model=` on every call — nothing resolves a model for you.
- You must pass `instructions=` (system prompt) on every call if you want one — there's no stored configuration.
- There are no tools (e.g. `web_search`) unless you attach them yourself on that specific request.
- There's no separate "resource" to create, version, or delete — it's just a model invocation, not a managed entity with its own identity/lifecycle.

### Step 1: Creating instance of AIProjectClient

In [ ]:
import os
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

PROJECT_ENDPOINT = os.environ["PROJECT_ENDPOINT"]
MODEL_DEPLOYMENT_NAME = os.environ.get("MODEL_DEPLOYMENT_NAME", "gpt-5")

project_client = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
)

project_client instance is created from the AIProjectClient class 

 project_client (AIProjectClient) is an Azure management-style client, generated from Azure's REST API conventions (azure-core
  transport). Its methods map to Foundry's control-plane endpoints: agents.get, agents.create_version, deployments.list, datasets,
  connections, etc. 
  
  It's purely CRUD-style mapping onto resources (agents, deployments, datasets, connections...). It has zero knowledge of
  conversation/runtime concepts because those aren't resources in that REST API at all  there's no POST /agents/x/messages endpoint for
  it to map to.

  Foundry's management REST API. E.g. roughly:

  - agents.get("careful-agent-82lybqkdrb")   →  GET  {endpoint}/agents/careful-agent-82lybqkdrb
  - agents.create_version("x", definition=…) →  POST {endpoint}/agents/x/versions
  - agents.delete("x")                        →  DELETE {endpoint}/agents/x
  

#### Step 2: Management Plane vs. Conversation Plane 

  AIProjectClient ≠ a Chat Client
  
  An instance of AIProjectClient has no concept of responses.create(...), streaming, chat messages, tool-call deltas , none of that. It literally
  cannot send a conversational turn; that's not a verb this client's generated API exposes

  The OpenAI responses API is a completely separate, much bigger surface (owned/maintained by the openai package) that knows how to do
  turns, streaming, structured outputs, function-calling, etc. Foundry's agent/model endpoints are designed to speak that protocol over
  HTTP, not Azure's REST conventions.


  So get_openai_client() is just a convenience factory: it takes the endpoint + credential you already gave AIProjectClient, and uses
  them to construct a properly-authenticated openai.OpenAI instance pointed at the right base_url (either the project's model endpoint,
  or a specific agent's endpoint if you pass agent_name=). 

Here are speaking about Control Plane (AIProjectClient) vs. Data Plane (openai.OpenAI)

https://learn.microsoft.com/en-us/python/api/azure-ai-projects/azure.ai.projects.aio.aiprojectclient?view=azure-python#azure-ai-projects-aio-aiprojectclient-get-openai-client

In [9]:
model_client = project_client.get_openai_client()

#### Step 3: Fetching responses

In [8]:
response = model_client.responses.create(
    model=MODEL_DEPLOYMENT_NAME,          # required - no agent to resolve this for you
    instructions="You are a concise assistant.",  # required if you want a system prompt - nothing is stored server-side
    input="What is Microsoft Foundry in one sentence?",
)

print(response.output_text)

Microsoft Foundry (Azure AI Foundry) is Microsoft’s Azure platform for designing, evaluating, securing, and deploying generative AI applications and copilots end to end.


----------------------------------------

# Calling an existing agent or creating a new one


#### Step 1: Calling the existing agent

In [ ]:
EXISTING_AGENT_NAME = os.environ.get("AGENT_NAME") or None

a. Connect to Project endpoint

In [17]:
# allow_preview=True is required to call an agent's endpoint directly via get_openai_client()
project_client = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
    allow_preview=True,
)

print("Connected to project:", PROJECT_ENDPOINT)

Connected to project: https://anees-mshf24ld-centralus.services.ai.azure.com/api/projects/anees-mshf24ld-centralus-project


b. Fetch the agent details

In [ ]:
from azure.ai.projects.models import PromptAgentDefinition

if EXISTING_AGENT_NAME:
    agent = project_client.agents.get(EXISTING_AGENT_NAME)
    print("Reusing existing agent:", agent.name)
else:
    definition = PromptAgentDefinition(
        model=MODEL_DEPLOYMENT_NAME,
        instructions="You are a concise, helpful assistant.",
    )
    version = project_client.agents.create_version(
        "foundry-tutorial-agent",
        definition=definition,
    )

Reusing existing agent: careful-agent-82lybqkdrb


- PromptAgentDefinition defines if it is a "prompt" vs "hosted" vs "workflow"
- create_version checks if if "foundry-tutorial-agent" doesn't exist yet, it creates the agent and its first version in one request

In [20]:
# # If creating a new agent, fetch it to get its name and other details. If reusing an existing agent, this step is not needed.
# agent = project_client.agents.get(version.name)
# print("Created new agent:", agent.name)

c. Call the agent

`get_openai_client(agent_name=...)` returns a standard OpenAI client pointed at the agent's own endpoint, so you interact with it using the normal `responses` API.

In [21]:
agent_name = agent.name if EXISTING_AGENT_NAME else version.name

agent_client = project_client.get_openai_client(agent_name=agent_name)

agent_response = agent_client.responses.create(
    input="What is Microsoft Foundry in one sentence?",
)

print(agent_response.output_text)

Microsoft Foundry is Microsoft’s unified enterprise AI platform for building, grounding, and governing AI apps and agents across their full lifecycle with open development and built-in security, compliance, and policy controls. ([azure.microsoft.com](https://azure.microsoft.com/en-us/products/ai-foundry/))


### Step 2: Multi-turn conversation

Pass `previous_response_id` to continue the same conversation with context.

In [22]:
follow_up = agent_client.responses.create(
    input="Can you say that again, but in French?",
    previous_response_id=response.id,
)

print(follow_up.output_text)

Microsoft Foundry (Azure AI Foundry) est la plateforme Azure de Microsoft pour concevoir, évaluer, sécuriser et déployer des applications d’IA générative et des copilotes de bout en bout.


------------------------------------

## Model call vs. Agent call

| | Direct model call (this notebook) | Foundry agent (`foundry_agent_sdk_demo.ipynb`) |
|---|---|---|
| What you call | `project_client.get_openai_client()` (no `agent_name`) | `project_client.get_openai_client(agent_name=...)` |
| Identity | None — just a model | Named, versioned resource (`agents.get` / `create_version` / `delete`) |
| Instructions | Passed on every request, or omitted | Stored once in the agent's definition |
| Tools (e.g. `web_search`) | None unless attached per-request | Attached once in the agent's definition |
| Reuse across apps | Re-specify model + instructions + tools every time | Just reference the agent by name |
| `allow_preview` needed? | No | Yes, to target an agent's own endpoint |

Same underlying `responses.create(...)` call shape either way — the agent is really just a named, versioned, server-side bundle of "model + instructions + tools" so callers don't have to repeat all three every time.